# 03 - Authoring Custom Physical Models & Parametrization

In this tutorial, you will learn how to:

1. Subclass `ModelComponent` to create brand new physical models.
2. Define custom 1-D and 2-D empirical lookup curves (`Curve1D`, `Surface2D`).
3. Embed an `OperatingStateMachine` to model start-up durations, warm-up purges, and ramp limits.
4. Modify and customize parameters of existing catalog models.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from energysim import world
from energysim.models.common.base_component import ModelComponent
from energysim.models.common.curve import Curve1D
from energysim.models.common.state_machine import OperatingStateMachine, OperatingState

print("Dependencies loaded!")


## Step 1: Create a Custom Supercapacitor Component

We implement a fast-discharge supercapacitor module with voltage limits and internal resistance:


In [ ]:
class SupercapacitorModule(ModelComponent):
    def __init__(self, name="supercap", capacitance_f=600.0, v_rated=48.0, r_esr=0.008, v_init=48.0, **kwargs):
        super().__init__(name=name, **kwargs)
        self.c_f = capacitance_f
        self.v_rated = v_rated
        self.r_esr = r_esr
        self.v_cell = v_init
        
        self.register_variables(
            inputs=["P_cmd_kW"],
            outputs=["P_actual_kW", "V_terminal", "SoC", "energy_kJ"]
        )

    def step_model(self, dt: float, time: float) -> None:
        p_cmd_w = float(self._inputs_val.get("P_cmd_kW", 0.0)) * 1000.0
        
        # Calculate current
        i = p_cmd_w / max(self.v_cell, 1.0)
        
        # dV = I * dt / C
        dv = (i / self.c_f) * dt
        self.v_cell = np.clip(self.v_cell + dv, 5.0, self.v_rated)
        
        v_term = self.v_cell + i * self.r_esr
        p_act_kw = (v_term * i) / 1000.0
        soc = (self.v_cell / self.v_rated) ** 2
        energy_kj = 0.5 * self.c_f * (self.v_cell ** 2) / 1000.0
        
        self._outputs_val["P_actual_kW"] = p_act_kw
        self._outputs_val["V_terminal"] = v_term
        self._outputs_val["SoC"] = soc
        self._outputs_val["energy_kJ"] = energy_kj

print("SupercapacitorModule class defined!")


## Step 2: Test Custom Model in energysim


In [ ]:
cap = SupercapacitorModule(name="fast_cap", capacitance_f=1200.0, v_rated=48.0)

sim = world(start_time=0, stop_time=300, t_macro=5)  # 5-minute study with 5s time-steps
sim.add_model("fast_cap", model=cap, outputs=["P_actual_kW", "V_terminal", "SoC"])

# Rapid charge and discharge sequence
def pulse_cmd(t):
    if 20 <= t < 80:
        return 15.0   # Fast charge 15 kW
    elif 150 <= t < 220:
        return -20.0  # Deep discharge 20 kW
    return 0.0

sim.add_signal("pulse", signal=pulse_cmd)
sim.add_connections({"pulse.y": "fast_cap.P_cmd_kW"})

sim.simulate(dashboard=False)
df_cap = sim.results(dashboard=False)["fast_cap"]
display(df_cap.head())
